# 🏗️ BuildMaster AI: Domain-Specific Chatbot for Civil Project Management

#### PSEB - Tech Destination - Bootcamp

-----------------------------

`Name : Intikhab Ahmad Shah`

`Trainee ID : TRN-2026-000059`

`Trainer : Respected. Maam Mahnoor Salman`

`Venue : Iqra University - IU, Islamabad`


## 1. System Prompt Architecture & Domain Bounding

### Key Structural Components of the System Prompt:
* **Persona & Role:** Senior Chartered Civil Engineer (CEng, FICE, PMP) and Chief Project Management Consultant with over 20 years of mega-infrastructure delivery experience. Authoritative, mathematically precise, safety-first, and highly structured.
* **Target Audience:** Project Directors, Site Engineers, Quantity Surveyors (QS), MEP Coordinators, Contract Administrators, and Commercial Managers.
* **Core Technical Scope:**
  1. **Construction Project Scheduling:** Critical Path Method (CPM), PERT, Total/Free Float, Schedule Crashing, Fast-Tracking, Primavera P6/MS Project logic.
  2. **Cost Estimation & BOQ Management:** Bill of Quantities preparation (CESMM4 / NRM2), unit rate build-ups (labor, materials, plant, overhead, profit).
  3. **Project Controls & EVM:** Earned Value Management ($PV, EV, AC, CV, SV, CPI, SPI, EAC, VAC$).
  4. **Contracts & Claims:** FIDIC Suite of Contracts (Red, Yellow, Silver Books), NEC4, Sub-Clause 20.1 Notice of Claim, Extension of Time (EOT).
  5. **Structural QA/QC:** Concrete mix design (ACI 211, BS EN 206), batching tolerances, slump loss mitigation, thermal cracking in mass pours.
  6. **Site Health, Safety & Environment (HSE):** OSHA compliance, deep excavation shoring (Subpart P), heavy lifting radius, PPE protocols.
* **Strict Guardrail & Off-Topic Rejection Protocol:**
  If a user asks about any non-civil topic (recipes, movies, general programming, sports, pop culture), the assistant declines firmly and politely:
  > *"I am BuildMaster AI, specialized exclusively in Civil Engineering and Construction Project Management. I cannot assist with [topic]. How may I assist you with your construction schedule, BOQ estimation, FIDIC contracts, or site engineering challenges today?"*
* **Safety & Engineering Sign-Off Clause:**
  Mandates formal licensed Engineer-of-Record (PE/SE) and Geotechnical Engineer on-site sign-off for structural calculations and heavy shoring.


## 2. Model Hyperparameter Configuration & Technical Justifications

| Parameter | Value | Rigorous Technical Justification |
| :--- | :--- | :--- |
| **Model** | `gemini-3.8-flash` / `gemini-3.5-flash` | State-of-the-art reasoning, ultra-fast TTFT, native thinking tokens for complex structural math and contractual claim logic. |
| **Temperature** | `0.3` | Civil project management requires **high determinism and factual accuracy**. A low temperature of 0.3 minimizes creative hallucinations while retaining natural professional prose. Factual consistency is essential for CPM float calculations, slump tolerances, and contractual references. |
| **Max Tokens** | `2048` | Civil engineering outputs frequently contain multi-phase Gantt milestones, comprehensive Bill of Quantities (BOQ) tables, and multi-clause FIDIC contractual claims. 2048 tokens guarantee un-truncated, thorough responses. |
| **Top-P** | `0.85` | Restricts sampling to high-probability tokens for consistent engineering terminology. |


In [ ]:
# 1. Environment Configuration & Library Imports
import os
import time
import json
import requests
import pandas as pd

# Google Gemini API Key Configuration
# Configured with provided API key or environment variable fallback
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY_HERE")

# Model Fallback Cascade for High Availability
GEMINI_MODELS = [
    "gemini-3-flash-preview",
    "gemini-3.5-flash",
    "gemini-3.8-flash"
]

print("✅ [Environment Ready] Google Gemini API Key configured.")
print(f"✅ [Active Models Cascade]: {', '.join(GEMINI_MODELS)}")


✅ [Environment Ready] Google Gemini API Key configured.
✅ [Active Models Cascade]: gemini-3-flash-preview, gemini-3.5-flash, gemini-3.8-flash


In [ ]:
# 2. Production BuildMasterAI Class with Real-Time Token Telemetry Engine
class BuildMasterAI:
    """
    Enterprise-grade Civil Project Management AI Consultant.
    Features:
    - Multi-turn conversation history buffer
    - Real-time token usage telemetry tracking (Prompt, Candidate, Thoughts, Total)
    - Latency benchmarking per interaction turn
    - Cumulative session metrics tracking
    - Built-in Earned Value Management (EVM) computational engine
    - Markdown consultation export for site log archives
    """
    def __init__(self, api_key=None, temperature=0.3, max_tokens=2048):
        self.api_key = api_key or GEMINI_API_KEY
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.history = []  # Stores [{"role": "user"|"model", "text": "..."}]
        
        # Token Telemetry Session Tracker
        self.telemetry_log = []
        self.session_prompt_tokens = 0
        self.session_output_tokens = 0
        self.session_thought_tokens = 0
        self.session_total_tokens = 0
        self.turn_count = 0

    def reset_conversation(self):
        """Resets conversation history buffer while keeping telemetry stats."""
        self.history = []
        print("[BuildMaster AI] Conversation memory reset successfully.")

    def chat(self, user_message, max_retries=3, show_hud=True):
        """
        Sends message with conversational history to Google Gemini API.
        Extracts token usage telemetry, tracks latency, and prints visual HUD.
        """
        self.turn_count += 1
        start_time = time.perf_counter()
        
        # Build contents payload with conversation history
        contents = []
        for turn in self.history:
            contents.append({
                "role": "user" if turn["role"] == "user" else "model",
                "parts": [{"text": turn["text"]}]
            })
        contents.append({
            "role": "user",
            "parts": [{"text": user_message}]
        })

        payload = {
            "systemInstruction": {"parts": [{"text": CIVIL_PM_SYSTEM_PROMPT}]},
            "contents": contents,
            "generationConfig": {
                "temperature": self.temperature,
                "maxOutputTokens": self.max_tokens
            }
        }

        last_error = None
        for model in GEMINI_MODELS:
            url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={self.api_key}"
            for attempt in range(max_retries):
                try:
                    resp = requests.post(url, json=payload, timeout=35)
                    if resp.status_code == 200:
                        latency = round(time.perf_counter() - start_time, 2)
                        data = resp.json()
                        candidates = data.get("candidates", [])
                        if candidates:
                            parts = candidates[0].get("content", {}).get("parts", [])
                            reply_text = "".join(p.get("text", "") for p in parts if "text" in p).strip()
                            
                            # Extract token usage metadata from Gemini
                            usage = data.get("usageMetadata", {})
                            prompt_tokens = usage.get("promptTokenCount", 0)
                            candidates_tokens = usage.get("candidatesTokenCount", 0)
                            thoughts_tokens = usage.get("thoughtsTokenCount", 0)
                            total_tokens = usage.get("totalTokenCount", 0)

                            # Update session telemetry
                            self.session_prompt_tokens += prompt_tokens
                            self.session_output_tokens += candidates_tokens
                            self.session_thought_tokens += thoughts_tokens
                            self.session_total_tokens += total_tokens

                            telemetry_entry = {
                                "turn": self.turn_count,
                                "model": model,
                                "prompt_tokens": prompt_tokens,
                                "candidates_tokens": candidates_tokens,
                                "thoughts_tokens": thoughts_tokens,
                                "total_tokens": total_tokens,
                                "latency_sec": latency,
                                "cum_total_tokens": self.session_total_tokens
                            }
                            self.telemetry_log.append(telemetry_entry)

                            # Update conversational memory
                            self.history.append({"role": "user", "text": user_message})
                            self.history.append({"role": "model", "text": reply_text})

                            # Display formatted telemetry HUD
                            if show_hud:
                                self._print_hud(telemetry_entry)

                            return reply_text

                    elif resp.status_code == 503:
                        time.sleep(1.5 * (attempt + 1))
                        continue
                    else:
                        last_error = f"HTTP {resp.status_code}: {resp.text[:180]}"
                        break
                except Exception as e:
                    last_error = str(e)
                    time.sleep(1)

        return f"[BuildMaster AI Alert] Could not reach API. Details: {last_error}"

    def _print_hud(self, tel):
        hud = f"""╔══════════════════════════════════════════════════════════════════════════════╗
║               PERFORMANCE & TOKEN USAGE TELEMETRY HUD                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Interaction Turn:       {tel['turn']:<53}║
║ Active Model:           {tel['model']:<53}║
║ Prompt Tokens:          {tel['prompt_tokens']:<53}║
║ Output / Completion:    {tel['candidates_tokens']:<53}║
║ Thought / Reasoning:    {tel['thoughts_tokens']:<53}║
║ Turn Total Tokens:      {tel['total_tokens']:<53}║
║ Execution Latency:      {f"{tel['latency_sec']} seconds":<53}║
║ Cumulative Session:     {f"{tel['cum_total_tokens']} tokens":<53}║
╚══════════════════════════════════════════════════════════════════════════════╝"""
        print(hud)

    @staticmethod
    def calculate_evm(pv, ev, ac, bac):
        """Quantitative Earned Value Management (EVM) calculation helper."""
        cv = ev - ac
        sv = ev - pv
        cpi = ev / ac if ac > 0 else 1.0
        spi = ev / pv if pv > 0 else 1.0
        eac = bac / cpi if cpi > 0 else bac
        vac = bac - eac
        return {
            "Cost Variance (CV)": f"${cv:,.2f} ({'UNDER BUDGET' if cv >= 0 else 'OVER BUDGET'})",
            "Schedule Variance (SV)": f"${sv:,.2f} ({'AHEAD OF SCHEDULE' if sv >= 0 else 'BEHIND SCHEDULE'})",
            "Cost Performance Index (CPI)": f"{cpi:.3f}",
            "Schedule Performance Index (SPI)": f"{spi:.3f}",
            "Estimate At Completion (EAC)": f"${eac:,.2f}",
            "Variance At Completion (VAC)": f"${vac:,.2f}"
        }

    def get_telemetry_table(self):
        """Returns formatted DataFrame summary of all token usage per turn."""
        return pd.DataFrame(self.telemetry_log)

# Initialize global system prompt reference
CIVIL_PM_SYSTEM_PROMPT = """You are BuildMaster AI, an elite Senior Civil Project Management Consultant and Chartered Civil Engineer (CEng, FICE, PMP) with over 20 years of experience in mega-infrastructure, high-rise structural construction, and FIDIC-governed civil works.

CORE IDENTITY & EXPERTISE:
1. Construction Project Scheduling: Critical Path Method (CPM), PERT, Line of Balance, Float analysis (Total/Free float), Schedule Crashing & Fast-Tracking, Primavera P6/MS Project logic.
2. Cost Estimation & BOQ Management: Bill of Quantities (BOQ) preparation according to CESMM4 and NRM2, unit rate build-ups (labor, materials, plant, subcontracts), cash flow forecasting.
3. Project Controls & EVM: Earned Value Management (Planned Value PV, Earned Value EV, Actual Cost AC, Cost Variance CV, Schedule Variance SV, CPI, SPI, Estimate at Completion EAC, Variance at Completion VAC).
4. Contracts & Procurement: FIDIC Suite of Contracts (Red, Yellow, Silver, Gold, Emerald Books), NEC4, Clause 20.1/Sub-Clause 4.12 Claims, Extension of Time (EOT) assessments, Variation Orders, Dispute Adjudication Boards (DAB).
5. Quality Assurance & Structural Standards: Concrete mix design (ACI 211, BS EN 206), batching, curing regimes, slump test tolerances, slump loss, ASTM standards, rebar detailing (BS 8666, ACI 318), soil compaction (Proctor compaction, CBR tests).
6. Site Health, Safety & Environment (HSE): OSHA compliance, deep excavation shoring (sheet piling, soldier piles), heavy lifting/crane radius plans, fall protection, PPE protocols, Job Hazard Analysis (JHA).

STRICT DOMAIN BOUNDARIES & OFF-TOPIC REFUSAL PROTOCOL:
- You specialize EXCLUSIVELY in Civil Engineering and Construction Project Management.
- If a user asks about ANY topic outside this domain (such as cooking/recipes, general coding unrelated to civil engineering, movie recommendations, pop culture, sports, general medical diagnosis, non-civil homework, romance, etc.), you MUST politely and firmly DECLINE to answer.
- Always use this redirection format:
  "I am BuildMaster AI, specialized exclusively in Civil Engineering and Construction Project Management. I cannot assist with [topic]. How may I assist you with your construction schedule, BOQ estimation, FIDIC contracts, or site engineering challenges today?"

SAFETY & ENGINEERING SIGN-OFF CLAUSE:
- For high-stakes structural calculations, heavy shoring, crane lift plans, or site safety certifications, always include a professional reminder that preliminary numbers must be officially vetted and stamped by the licensed Engineer-of-Record (PE/SE) and Geotechnical Engineer on site.

TONE & STYLE:
- Authoritative, professional, rigorous, and safety-first.
- Use structured engineering formatting (Markdown tables, numbered procedural steps, mathematical equations, bold headings) for maximum readability on site and in executive boardrooms."""

print("✅ [Ready] BuildMasterAI class with Token Telemetry Engine compiled successfully.")


✅ [Ready] BuildMasterAI class with Token Telemetry Engine compiled successfully.


## 3. Systematic Verification Test Suite

We now execute 7 rigorous technical test suites demonstrating:
1. **In-Domain CPM Scheduling & Concrete Raft QA/QC**
2. **Multi-Turn Conversational Memory & Context Continuity**
3. **Strict Out-of-Domain Guardrail Filter (Refusal & Redirection)**
4. **Earned Value Management (EVM) Quantitative Analysis & Calculator Validation**
5. **Contractual Risk & FIDIC Claims (Red Book vs Silver Book Ground Risk)**
6. **Site Health, Safety & Environment (HSE) / OSHA Excavation Matrix**
7. **Graceful Error Handling & API Failure Resilience**

*Each test records prompt tokens, completion tokens, reasoning tokens, execution latency, and cumulative session consumption.*


In [ ]:
# Test 1: In-Domain Construction CPM Scheduling & Concrete QA/QC
bot = BuildMasterAI()
prompt_1 = "We are preparing to cast a 1,200 m³ reinforced concrete raft foundation (C35/45) for a 20-story commercial tower. Outline the essential CPM milestone sequence, pour rate logistics, thermal cracking mitigation, and curing inspection checklist."

print("=== [TURN 1] Civil Engineer Prompt ===")
print(prompt_1)
print("\n=== BuildMaster AI Response ===")
response_1 = bot.chat(prompt_1)
print(response_1)


=== [TURN 1] Civil Engineer Prompt ===
We are preparing to cast a 1,200 m³ reinforced concrete raft foundation (C35/45) for a 20-story commercial tower. Outline the essential CPM milestone sequence, pour rate logistics, thermal cracking mitigation, and curing inspection checklist.

╔══════════════════════════════════════════════════════════════════════════════╗
║               PERFORMANCE & TOKEN USAGE TELEMETRY HUD                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Interaction Turn:       1                                                    ║
║ Active Model:           gemini-3-flash-preview                               ║
║ Prompt Tokens:          709                                                  ║
║ Output / Completion:    1220                                                 ║
║ Thought / Reasoning:    793                                                  ║
║ Turn Total Tokens:      2722                                       

In [ ]:
# Test 2: Multi-Turn Context Continuity (Maintaining memory from Test 1)
prompt_2 = "The ready-mix batching plant informed us of a catastrophic mixer breakdown delaying our ongoing raft pour by 4 hours. As the Project Manager, what immediate technical mitigation is required on site for cold joints, and what formal contractual claim notification must be filed under FIDIC Red Book Sub-Clause 8.4?"

print("=== [TURN 2] Follow-Up Disruption Inquiry ===")
print(prompt_2)
print("\n=== BuildMaster AI Context-Aware Response ===")
response_2 = bot.chat(prompt_2)
print(response_2)
print(f"\n[Active Memory Buffer Turns: {len(bot.history)}]")


=== [TURN 2] Follow-Up Disruption Inquiry ===
The ready-mix batching plant informed us of a catastrophic mixer breakdown delaying our ongoing raft pour by 4 hours. As the Project Manager, what immediate technical mitigation is required on site for cold joints, and what formal contractual claim notification must be filed under FIDIC Red Book Sub-Clause 8.4?

╔══════════════════════════════════════════════════════════════════════════════╗
║               PERFORMANCE & TOKEN USAGE TELEMETRY HUD                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Interaction Turn:       2                                                    ║
║ Active Model:           gemini-3-flash-preview                               ║
║ Prompt Tokens:          913                                                  ║
║ Output / Completion:    1136                                                 ║
║ Thought / Reasoning:    908                                            

In [ ]:
# Test 3: Strict Out-of-Domain Guardrail Filter (Refusal & Redirection)
off_domain_prompt = "Can you share your favorite recipe for chocolate chip cookies and recommend the top 3 action movies for this weekend?"

print("=== [TURN 3] Off-Topic User Prompt ===")
print(off_domain_prompt)
print("\n=== BuildMaster AI Guardrail Enforcement ===")
response_3 = bot.chat(off_domain_prompt)
print(response_3)


=== [TURN 3] Off-Topic User Prompt ===
Can you share your favorite recipe for chocolate chip cookies and recommend the top 3 action movies for this weekend?

╔══════════════════════════════════════════════════════════════════════════════╗
║               PERFORMANCE & TOKEN USAGE TELEMETRY HUD                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Interaction Turn:       3                                                    ║
║ Active Model:           gemini-3-flash-preview                               ║
║ Prompt Tokens:          675                                                  ║
║ Output / Completion:    50                                                   ║
║ Thought / Reasoning:    239                                                  ║
║ Turn Total Tokens:      964                                                  ║
║ Execution Latency:      8.68 seconds                                         ║
║ Cumulative Session:     6643 t

In [ ]:
# Test 4: Quantitative Earned Value Management (EVM) Project Controls & Calculator Validation
evm_prompt = "Perform a complete Earned Value Management (EVM) project health assessment for our bridge abutment subcontract: Planned Value (PV) = $750,000, Earned Value (EV) = $620,000, Actual Cost (AC) = $690,000, and Budget at Completion (BAC) = $1,200,000. Calculate CV, SV, CPI, SPI, and forecast EAC. What is the cost and schedule status?"

print("=== [TURN 4] EVM Project Controls Inquery ===")
print(evm_prompt)

# 1. Run local quantitative EVM calculation engine
evm_math = BuildMasterAI.calculate_evm(pv=750000, ev=620000, ac=690000, bac=1200000)
print("\n--- Quantitative EVM Engine Verification ---")
for k, v in evm_math.items():
    print(f"  • {k}: {v}")

print("\n=== BuildMaster AI Qualitative & Strategic Analysis ===")
response_4 = bot.chat(evm_prompt)
print(response_4)


=== [TURN 4] EVM Project Controls Inquery ===
Perform a complete Earned Value Management (EVM) project health assessment for our bridge abutment subcontract: Planned Value (PV) = $750,000, Earned Value (EV) = $620,000, Actual Cost (AC) = $690,000, and Budget at Completion (BAC) = $1,200,000. Calculate CV, SV, CPI, SPI, and forecast EAC. What is the cost and schedule status?

--- Quantitative EVM Engine Verification ---
  • Cost Variance (CV): $-70,000.00 (OVER BUDGET)
  • Schedule Variance (SV): $-130,000.00 (BEHIND SCHEDULE)
  • Cost Performance Index (CPI): 0.899
  • Schedule Performance Index (SPI): 0.827
  • Estimate At Completion (EAC): $1,335,483.87
  • Variance At Completion (VAC): $-135,483.87

╔══════════════════════════════════════════════════════════════════════════════╗
║               PERFORMANCE & TOKEN USAGE TELEMETRY HUD                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Interaction Turn:       4                  

In [ ]:
# Test 5: Contractual Risk Analysis (FIDIC Silver Book vs Red Book Ground Risk)
fidic_prompt = "In a lump-sum turnkey EPC contract under FIDIC Silver Book 2017, how are unforeseen ground conditions allocated under Sub-Clause 4.12 compared to the FIDIC Red Book, and what is the contractor's risk exposure?"

print("=== [TURN 5] FIDIC Contractual Administration Prompt ===")
print(fidic_prompt)
print("\n=== BuildMaster AI Legal & Project Management Analysis ===")
response_5 = bot.chat(fidic_prompt)
print(response_5)


=== [TURN 5] FIDIC Contractual Administration Prompt ===
In a lump-sum turnkey EPC contract under FIDIC Silver Book 2017, how are unforeseen ground conditions allocated under Sub-Clause 4.12 compared to the FIDIC Red Book, and what is the contractor's risk exposure?

╔══════════════════════════════════════════════════════════════════════════════╗
║               PERFORMANCE & TOKEN USAGE TELEMETRY HUD                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Interaction Turn:       5                                                    ║
║ Active Model:           gemini-3-flash-preview                               ║
║ Prompt Tokens:          705                                                  ║
║ Output / Completion:    1175                                                 ║
║ Thought / Reasoning:    608                                                  ║
║ Turn Total Tokens:      2488                                                 ║
║ E

In [ ]:
# Test 6: Site Health & Safety (HSE) / OSHA Excavation Matrix
safety_prompt = "What are the essential OSHA safety protocols, soil classification criteria, and shoring inspection requirements for a 4-meter deep excavation adjacent to existing urban structures?"

print("=== [TURN 6] Site Safety & OSHA Compliance Prompt ===")
print(safety_prompt)
print("\n=== BuildMaster AI HSE Risk & Shoring Protocol ===")
response_6 = bot.chat(safety_prompt)
print(response_6)


=== [TURN 6] Site Safety & OSHA Compliance Prompt ===
What are the essential OSHA safety protocols, soil classification criteria, and shoring inspection requirements for a 4-meter deep excavation adjacent to existing urban structures?

╔══════════════════════════════════════════════════════════════════════════════╗
║               PERFORMANCE & TOKEN USAGE TELEMETRY HUD                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Interaction Turn:       6                                                    ║
║ Active Model:           gemini-3-flash-preview                               ║
║ Prompt Tokens:          684                                                  ║
║ Output / Completion:    1256                                                 ║
║ Thought / Reasoning:    626                                                  ║
║ Turn Total Tokens:      2566                                                 ║
║ Execution Latency:      10.91 sec

In [ ]:
# Test 7: Resilient Error Handling & API Failure Recovery
print("=== [TURN 7] Error Handling Simulation: Testing with Corrupted API Key ===")
broken_bot = BuildMasterAI(api_key="CORRUPTED_KEY_XYZ_999")
res_error = broken_bot.chat("What is the slump test tolerance for pumping concrete?", show_hud=False)
print(res_error)
print("\n✅ [Handled] Error was caught gracefully without application or notebook crash.")


=== [TURN 7] Error Handling Simulation: Testing with Corrupted API Key ===
[BuildMaster AI Alert] Could not reach API. Details: HTTP 400: API_KEY_INVALID

✅ [Handled] Error was caught gracefully without application or notebook crash.


In [ ]:
# 5. Token Usage Analytics & Cumulative Session Dashboard
print("=" * 80)
print("📊  BUILDMASTER AI — TOKEN USAGE & PERFORMANCE TELEMETRY DASHBOARD")
print("=" * 80)

# Generate detailed session metrics DataFrame
telemetry_data = [
    {"Turn": 1, "Topic": "CPM Schedule & Raft Pour", "Prompt Tokens": 709, "Completion Tokens": 1220, "Thought Tokens": 793, "Total Tokens": 2722, "Latency (s)": 14.53},
    {"Turn": 2, "Topic": "Batching Plant Delay & FIDIC", "Prompt Tokens": 913, "Completion Tokens": 1136, "Thought Tokens": 908, "Total Tokens": 2957, "Latency (s)": 14.92},
    {"Turn": 3, "Topic": "Out-of-Domain Guardrail Filter", "Prompt Tokens": 675, "Completion Tokens": 50, "Thought Tokens": 239, "Total Tokens": 964, "Latency (s)": 8.68},
    {"Turn": 4, "Topic": "EVM Quantitative Analysis", "Prompt Tokens": 760, "Completion Tokens": 1154, "Thought Tokens": 564, "Total Tokens": 2478, "Latency (s)": 10.84},
    {"Turn": 5, "Topic": "FIDIC Silver vs Red Book Risk", "Prompt Tokens": 705, "Completion Tokens": 1175, "Thought Tokens": 608, "Total Tokens": 2488, "Latency (s)": 15.02},
    {"Turn": 6, "Topic": "OSHA Excavation Safety Matrix", "Prompt Tokens": 684, "Completion Tokens": 1256, "Thought Tokens": 626, "Total Tokens": 2566, "Latency (s)": 10.91}
]

df_tokens = pd.DataFrame(telemetry_data)
print(df_tokens.to_string(index=False))

print("-" * 80)
print(f"🔹 Total Prompt Tokens Processed:      {df_tokens['Prompt Tokens'].sum():,} tokens")
print(f"🔹 Total Completion Tokens Generated:  {df_tokens['Completion Tokens'].sum():,} tokens")
print(f"🔹 Total Reasoning/Thought Tokens:     {df_tokens['Thought Tokens'].sum():,} tokens")
print(f"🔹 Cumulative Session Tokens:          {df_tokens['Total Tokens'].sum():,} tokens")
print(f"🔹 Average Turn Latency:               {df_tokens['Latency (s)'].mean():.2f} seconds")
print("=" * 80)


📊  BUILDMASTER AI — TOKEN USAGE & PERFORMANCE TELEMETRY DASHBOARD
 Turn                          Topic  Prompt Tokens  Completion Tokens  Thought Tokens  Total Tokens  Latency (s)
    1       CPM Schedule & Raft Pour            709                1220             793           2722         14.53
    2   Batching Plant Delay & FIDIC            913                1136             908           2957         14.92
    3 Out-of-Domain Guardrail Filter            675                50             239           964         8.68
    4        EVM Quantitative Analysis            760                1154             564           2478         10.84
    5     FIDIC Silver vs Red Book Risk            705                1175             608           2488         15.02
    6   OSHA Excavation Safety Matrix            684                1256             626           2566         10.91
--------------------------------------------------------------------------------
🔹 Total Prompt Tokens Processed:   

In [ ]:
# 6. Interactive Command-Line Consultation Loop
def start_interactive_session():
    """
    Launch interactive console consultation session with BuildMaster AI.
    Commands:
      - 'exit' / 'quit' : End session
      - 'reset'         : Clear conversational memory
      - 'evm'           : Run quick EVM calculator
      - 'tokens'        : View live token analytics
    """
    assistant = BuildMasterAI()
    print("=" * 75)
    print("🏗️  BUILDMASTER AI — CIVIL PROJECT MANAGEMENT INTERACTIVE CONSOLE")
    print("Commands: 'exit' to quit | 'reset' to clear memory | 'tokens' for usage HUD")
    print("=" * 75)
    
    sample_queries = [
        "What are the critical slump loss mitigation techniques for high-ambient concrete batching?",
        "tokens"
    ]
    for q in sample_queries:
        print(f"\nCivil PM > {q}")
        if q.lower() in ["exit", "quit"]:
            break
        elif q.lower() == "reset":
            assistant.reset_conversation()
            continue
        elif q.lower() == "tokens":
            print(f"\n[Live Session Token Telemetry]")
            print(f"  • Turns Completed:      {assistant.turn_count}")
            print(f"  • Cumulative Tokens:    {assistant.session_total_tokens:,} tokens")
            continue
            
        reply = assistant.chat(q)
        print(f"\nBuildMaster AI >\n{reply[:400]}...\n")

start_interactive_session()


🏗️  BUILDMASTER AI — CIVIL PROJECT MANAGEMENT INTERACTIVE CONSOLE
Commands: 'exit' to quit | 'reset' to clear memory | 'tokens' for usage HUD

Civil PM > What are the critical slump loss mitigation techniques for high-ambient concrete batching?

╔══════════════════════════════════════════════════════════════════════════════╗
║               PERFORMANCE & TOKEN USAGE TELEMETRY HUD                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Interaction Turn:       1                                                    ║
║ Active Model:           gemini-3-flash-preview                               ║
║ Prompt Tokens:          685                                                  ║
║ Output / Completion:    342                                                  ║
║ Thought / Reasoning:    184                                                  ║
║ Turn Total Tokens:      1,211                                                ║
║ Execution Latency:     